<a href="https://colab.research.google.com/github/Arrow66/Arrow66/blob/main/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Collection notebook

**Research question:** Which characteristics of a company determine how much its share price falls when interest rates rise?


---


To answer the research question we need four data joined together:

| data needed  | why ?| Source |
|---|---|---|
| Share prices for a broad set of companies | To measure how each stock *actually* moved | Yahoo Finance |
| Interest rates, daily | whose movement we test against | FRED (US Federal Reserve) |
| Company characteristics |  debt, cash flow, valuation, sector | Yahoo Finance |
| Fama-French factors | To prove any rate effect is genuine, not a disguised size or value effect | Ken French Data Library, Dartmouth |

## notebook summary

Raw files in `data/raw/`, downloaded once and cached so that every later notebook reads
the **same** data and results stay reproducible:

1. `sp500_constituents.csv`” the company universe
2. `fred_*.csv` ” interest rate series
3. `fama_french_5factor_daily.csv`  control factors
4. `daily_prices.csv`  daily closing prices
5. `company_fundamentals.csv` balance sheet and valuation measures
6. `SOURCES.txt`  every URL with the date it was retrieved

## Reproducibility

Every source is public and needs **no API key and no account**. Raw downloads are saved
unchanged so the tables in the report can be regenerated exactly. Re-running this notebook
reuses the cached files instead of re-downloading; to force a fresh pull, set
`FORCE_REFRESH = True` in the configuration cell below.

## 1. Setup

Imports, then all configuration in a single cell so that every choice made in this
study is visible in one place and can be cited directly in the report.

In [13]:
import io
import time
import zipfile
import urllib.request
from datetime import date
from pathlib import Path

import pandas as pd
import yfinance as yf

print(f"pandas   {pd.__version__}")
print(f"yfinance {yf.__version__}")

pandas   2.2.2
yfinance 0.2.66


In [14]:
# =============================================================================
# CONFIGURATION â€” every choice made in this study lives here
# =============================================================================

# --- Where files go ----------------------------------------------------------
# Resolved from the working directory so the notebook runs whether Jupyter was started
# inside notebooks/ or at the project root.
_working_directory = Path.cwd()
PROJECT_ROOT = _working_directory.parent if _working_directory.name == "notebooks" else _working_directory
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

# --- Study window ------------------------------------------------------------
# Rate sensitivity is estimated over the recent high-rate period rather than a long
# history. A company's exposure reflects the balance sheet it has today, and stretching
# back further would blend in a different rate regime and different corporate finances.
PRICE_START = "2023-01-01"
PRICE_END = date.today().isoformat()

# --- Interest rate series (FRED) ---------------------------------------------
# DFII10 is the rate factor. It is the 10-year *inflation-protected* (real) yield,
# which is the right choice because the real yield is the discount rate that actually
# erodes the present value of future profits. A nominal yield rising purely because
# inflation expectations rose is a different economic event. The rest give context.
FRED_SERIES = {
    "DFII10": "10-year Treasury inflation-indexed (real) yield, percent",
    "DGS10": "10-year Treasury constant maturity (nominal) yield, percent",
    "DGS30": "30-year Treasury constant maturity (nominal) yield, percent",
    "T10YIE": "10-year breakeven inflation expectation, percent",
}
RATE_FACTOR = "DFII10"

# --- Source URLs -------------------------------------------------------------
FRED_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
FAMA_FRENCH_URL = (
    "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
    "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip"
)
SP500_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# --- Download behaviour ------------------------------------------------------
HTTP_TIMEOUT = 45          # seconds before a request is abandoned
MAX_RETRIES = 3            # transient network failures are common; retry before failing
RETRY_PAUSE = 5            # seconds to wait between retries

FUNDAMENTALS_PAUSE = 0.4   # seconds between company requests, to stay inside rate limits

# Set to True to ignore cached files and re-download everything from scratch.
FORCE_REFRESH = False

# --- Create the folders ------------------------------------------------------
for folder in (DATA_RAW, DATA_PROCESSED):
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw data     : {DATA_RAW}")
print(f"Study window : {PRICE_START} to {PRICE_END}")
print(f"Rate factor  : {RATE_FACTOR} â€” {FRED_SERIES[RATE_FACTOR]}")

Project root : /content
Raw data     : /content/data/raw
Study window : 2023-01-01 to 2026-08-16
Rate factor  : DFII10 â€” 10-year Treasury inflation-indexed (real) yield, percent


### Two small helpers

`download_bytes` retries a few times before giving up, because a single dropped
connection should not force a rerun of the whole notebook.

`is_cached` decides whether we already hold a file. Caching is what makes the study
reproducible: once downloaded, the analysis always runs against the same snapshot,
even though the live sources keep updating.

In [15]:
def download_bytes(url):
    """Fetch a URL and return its raw bytes, retrying on transient network failures."""
    last_error = None
    HTTP_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8" }
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            request = urllib.request.Request(url, headers=HTTP_HEADERS)
            with urllib.request.urlopen(request, timeout=HTTP_TIMEOUT) as response:
                return response.read()
        except Exception as error:
            last_error = error
            if attempt < MAX_RETRIES:
                print(f"   attempt {attempt} failed ({type(error).__name__}), retrying...")
                time.sleep(RETRY_PAUSE)

    raise RuntimeError(f"Could not download {url} after {MAX_RETRIES} attempts") from last_error


def is_cached(path):
    """True when a usable copy already exists and we are not forcing a refresh."""
    return path.exists() and path.stat().st_size > 0 and not FORCE_REFRESH


# Records every source we touch, written to SOURCES.txt at the end of the notebook.
source_log = []


def log_source(name, url, path, note=""):
    """Record where a file came from, so the report can document its provenance."""
    source_log.append({
        "dataset": name,
        "url": url,
        "saved_as": path.name,
        "retrieved_on": date.today().isoformat(),
        "note": note,
    })


print("Helpers ready.")

Helpers ready.


## 2. S&P500 company details fetch

In [16]:
constituents_file = DATA_RAW / "sp500_constituents.csv"

if is_cached(constituents_file):
    constituents = pd.read_csv(constituents_file)
    print(f"Using cached file ({len(constituents)} companies)")
else:
    print("Downloading S&P 500 constituent list from Wikipedia...")
    html = download_bytes(SP500_URL)

    # The page holds several tables; the first is the current constituent list.
    table = pd.read_html(io.BytesIO(html))[0]

    constituents = table.rename(columns={
        "Symbol": "ticker",
        "Security": "company_name",
        "GICS Sector": "sector",
        "GICS Sub-Industry": "sub_industry",
    })[["ticker", "company_name", "sector", "sub_industry"]]

    # Yahoo writes share classes with a dash where Wikipedia uses a dot (BRK.B -> BRK-B).
    constituents["ticker"] = constituents["ticker"].str.replace(".", "-", regex=False)

    constituents.to_csv(constituents_file, index=False)
    print(f"Saved {len(constituents)} companies to {constituents_file.name}")

log_source(
    "S&P 500 constituents",
    SP500_URL,
    constituents_file,
    "Current index membership; survivorship bias noted in the report.",
)

print(f"\nCompanies: {len(constituents)}")
print(f"Sectors:   {constituents['sector'].nunique()}\n")
print(constituents["sector"].value_counts().to_string())
constituents.head()

Using cached file (503 companies)

Companies: 503
Sectors:   11

sector
Industrials               83
Financials                76
Information Technology    73
Health Care               59
Consumer Discretionary    47
Consumer Staples          34
Utilities                 31
Real Estate               31
Materials                 25
Communication Services    23
Energy                    21


,ticker,company_name,sector,sub_industry
0,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,A. O. Smith,Industrials,Building Products
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,Accenture,Information Technology,IT Consulting & Other Services


## 3. Interest rates

From **FRED**, the US Federal Reserve's public data service. It serves plain CSV over a
normal web link with no key or account

### Why the *real* yield is the rate factor

`DFII10` is the 10-year Treasury Inflation-Protected yield means return an investor earns
**after** inflation.


The other three series are collected for context, for charts, and so the report can show
the nominal and real pictures side by side.

In [17]:
def fetch_fred_series(series_id):
    """Download one FRED series, or read it back from the cache if already held.

    Returns a DataFrame with a 'date' column and one value column named after the series.
    FRED marks days with no observation â€” weekends, holidays, market closures â€” with a
    '.' placeholder, which becomes NaN here and is handled in the cleaning notebook.
    """
    path = DATA_RAW / f"fred_{series_id}.csv"

    if is_cached(path):
        print(f"   {series_id:8s} cached")
    else:
        print(f"   {series_id:8s} downloading...")
        raw = download_bytes(FRED_URL.format(series_id=series_id))
        path.write_bytes(raw)

    series = pd.read_csv(path)
    series.columns = ["date", series_id]
    series["date"] = pd.to_datetime(series["date"])
    series[series_id] = pd.to_numeric(series[series_id], errors="coerce")

    log_source(f"FRED {series_id}", FRED_URL.format(series_id=series_id), path,
               FRED_SERIES[series_id])
    return series


print("Interest rate series:")
rate_frames = [fetch_fred_series(series_id) for series_id in FRED_SERIES]

# Combine into one table, one row per date, one column per series.
rates = rate_frames[0]
for frame in rate_frames[1:]:
    rates = rates.merge(frame, on="date", how="outer")

rates = rates.sort_values("date").reset_index(drop=True)

print(f"\nCombined rate table: {len(rates):,} rows, {rates['date'].min().date()} to {rates['date'].max().date()}")
print("\nMost recent values:")
print(rates.dropna(subset=[RATE_FACTOR]).tail(3).to_string(index=False))

Interest rate series:
   DFII10   cached
   DGS10    cached
   DGS30    cached
   T10YIE   cached

Combined rate table: 16,859 rows, 1962-01-02 to 2026-08-14

Most recent values:
      date  DFII10  DGS10  DGS30  T10YIE
2026-08-11    2.43   4.70   5.24    2.27
2026-08-12    2.42   4.68   5.24    2.26
2026-08-13    2.39   4.63   5.21    2.24


## 4. Fama-French factors for the control variables

 Suppose we find that a group of stocks falls hard when rates rise,
and those stocks happen to be small, expensive, unprofitable technology companies. Have we
discovered *rate sensitivity*? Or have we just rediscovered that small companies are volatile,
and confused one thing for another? Without controls we genuinely cannot tell.

Fama and French showed that a handful of characteristics explain most of the variation in
stock returns. By including them in the regression, whatever the rate factor explains is what
is left over **after** size, value, profitability and investment have taken their share. If the
rate effect survives that, it is real.

The five factors:

| Factor | description |
|---|---|
| `Mkt-RF` | The overall market's return above the risk-free rate |
| `SMB` | Small companies minus big ones (the size effect) |
| `HML` | Cheap companies minus expensive ones (the value effect) |
| `RMW` | Profitable companies minus unprofitable ones |
| `CMA` | Conservative investors minus aggressive spenders |
| `RF` | The risk-free rate, used to convert returns to excess returns |

`RMW` and `CMA` matter especially here: our hypothesis about negative free cash flow is close
to a profitability story, and our hypothesis about heavy capital spending is close to an
investment story. Controlling for both is what stops us claiming a rate effect that is
really one of these in disguise.

Source: the **Ken French Data Library** free to download

In [18]:
factors_file = DATA_RAW / "fama_french_5factor_daily.csv"

if is_cached(factors_file):
    print("Using cached Fama-French factors")
else:
    print("Downloading Fama-French 5 factors from the Ken French Data Library...")
    archive = zipfile.ZipFile(io.BytesIO(download_bytes(FAMA_FRENCH_URL)))
    csv_name = archive.namelist()[0]
    text = archive.read(csv_name).decode("latin-1")

    # The file opens with a few lines describing the CRSP database version before the
    # real column names, and closes with a copyright line after a blank separator.
    # We locate the header rather than assume a fixed offset, since the preamble
    # changes length whenever the library is updated.
    lines = text.splitlines()
    header_row = next(i for i, line in enumerate(lines) if "Mkt-RF" in line)

    data_lines = []
    for line in lines[header_row + 1:]:
        if not line.strip():          # blank line marks the end of the data block
            break
        data_lines.append(line)

    print(f"   header at line {header_row}, {len(data_lines):,} daily rows found")
    factors_file.write_text(
        "date,mkt_rf,smb,hml,rmw,cma,rf\n" + "\n".join(data_lines),
        encoding="utf-8",
    )

factors = pd.read_csv(factors_file)

# Dates arrive as YYYYMMDD integers, and the values are percentages, so 0.51 means 0.51%.
# Converting to decimals now keeps every later calculation in consistent units.
factors["date"] = pd.to_datetime(factors["date"], format="%Y%m%d")
for column in ["mkt_rf", "smb", "hml", "rmw", "cma", "rf"]:
    factors[column] = pd.to_numeric(factors[column], errors="coerce") / 100.0

factors = factors.sort_values("date").reset_index(drop=True)

log_source("Fama-French 5 factors (daily)", FAMA_FRENCH_URL, factors_file,
           "Converted from percent to decimal; see Fama & French (1993, 2015).")

# The library rebuilds from CRSP on a schedule, so the factors typically end a month or
# two before today. That lag sets the end of the estimation window in the modelling
last_factor_date = factors["date"].max()
lag_days = (pd.Timestamp.today().normalize() - last_factor_date).days

print(f"\n{len(factors):,} daily rows, {factors['date'].min().date()} to {last_factor_date.date()}")
print(f"Factors end {lag_days} days before today â€” this caps the estimation window.")
factors.tail(3)

Using cached Fama-French factors

15,854 daily rows, 1963-07-01 to 2026-06-30
Factors end 47 days before today â€” this caps the estimation window.


,date,mkt_rf,smb,hml,rmw,cma,rf
15851,2026-06-26,0.0015,0.0136,-0.0095,0.0044,0.0004,0.0001
15852,2026-06-29,0.0120,-0.0089,-0.0090,-0.0177,-0.0050,0.0001
15853,2026-06-30,0.0073,-0.0010,-0.0062,-0.0110,-0.0049,0.0001


## 5. Daily share prices

Roughly 500 companies from 2023 to today, via `yfinance`.

**Prices are split- and dividend-adjusted** (`auto_adjust=True`).
An unadjusted price series shows a 4-for-1 stock split as a 75% crash, which would be recorded
as a catastrophic loss that never happened and would poison the return calculations. Adjusting
also credits dividends back, so the series reflects what an investor actually earned.

We keep only the closing price. Highs, lows and volume are not needed to answer this question,
and dropping them keeps the file to a manageable size.

In [19]:
prices_file = DATA_RAW / "daily_prices.csv"
tickers = constituents["ticker"].tolist()

if is_cached(prices_file):
    prices = pd.read_csv(prices_file, parse_dates=["date"])
    print(f"Using cached prices ({prices['ticker'].nunique()} companies)")
else:
    print(f"Downloading prices for {len(tickers)} companies ({PRICE_START} to {PRICE_END})...")
    print("This takes a couple of minutes on the first run.\n")

    wide = yf.download(
        tickers,
        start=PRICE_START,
        end=PRICE_END,
        auto_adjust=True,     # adjust for splits and dividends
        progress=False,       # the per-ticker bar would bury this notebook in output
        threads=True,
    )["Close"]

    # Reshape from one column per company into one row per company per day, which is
    # the tidy layout every later step expects. Companies that were not yet listed on a
    # given day come through as NaN and are dropped rather than filled â€” an unlisted
    # company has no price, and inventing one would be a fabricated observation.
    prices = (
        wide.stack()
        .rename("close")
        .reset_index()
        .rename(columns={"Date": "date", "Ticker": "ticker"})
        .dropna(subset=["close"])
        .sort_values(["ticker", "date"])
        .reset_index(drop=True)
    )

    prices.to_csv(prices_file, index=False)
    print(f"\nSaved {len(prices):,} rows to {prices_file.name}")

log_source("Daily adjusted close prices", "https://finance.yahoo.com (via yfinance)",
           prices_file, f"Split- and dividend-adjusted, {PRICE_START} to {PRICE_END}.")

# How many trading days did each company actually provide? Companies with too few
# will be dropped in the cleaning notebook, because a regression on a short series
# produces an estimate too noisy to trust.
days_per_company = prices.groupby("ticker")["date"].count()

print(f"\nCompanies retrieved : {prices['ticker'].nunique()} of {len(tickers)}")
print(f"Date range          : {prices['date'].min().date()} to {prices['date'].max().date()}")
print(f"Trading days each   : median {days_per_company.median():.0f}, "
      f"minimum {days_per_company.min():.0f}")
print(f"Companies under 400 days: {(days_per_company < 400).sum()} (recent listings)")
prices.head()

Using cached prices (503 companies)

Companies retrieved : 503 of 503
Date range          : 2023-01-03 to 2026-08-14
Trading days each   : median 907, minimum 43
Companies under 400 days: 4 (recent listings)


,date,ticker,close
0,2023-01-03,A,146.134735
1,2023-01-04,A,147.722336
2,2023-01-05,A,148.150848
3,2023-01-06,A,143.826416
4,2023-01-09,A,143.631638


## 6. Stock splits — a safety net for the price data

The prices above are split-adjusted, but that adjustment is applied by Yahoo and occasionally
runs late. When a split is only days old, the historical prices sometimes have not been
restated yet, leaving a fake collapse in the data: Monster Beverage's two-for-one split on
11 August 2026 shows up as a 49.6% single-day loss.

So we collect the **actual split calendar** as well. The cleaning notebook uses it to tell a
real crash apart from an unadjusted split.

Guessing from the price move alone is not good enough, and it is worth saying why, because it
looks like it should work. A three-for-two split cuts the price by 33.3%, but Supermicro fell
32.7% the day its auditor resigned and The Trade Desk fell 33.0% on an earnings miss. Those are
genuine events that a purely arithmetic rule would silently erase. Checking against the real
calendar removes the guesswork.

In [20]:
splits_file = DATA_RAW / "stock_splits.csv"

if is_cached(splits_file):
    splits = pd.read_csv(splits_file, parse_dates=["date"])
    print(f"Using cached split calendar ({len(splits)} events)")
else:
    print("Downloading the split calendar for every company...")

    actions = yf.download(
        tickers,
        start=PRICE_START,
        end=PRICE_END,
        actions=True,        # adds dividend and split columns
        auto_adjust=True,
        progress=False,
        threads=True,
    )["Stock Splits"]

    # Most days for most companies are zero. Keep only the events.
    splits = (
        actions.stack()
        .rename("split_ratio")
        .reset_index()
        .rename(columns={"Date": "date", "Ticker": "ticker"})
    )
    splits = splits[splits["split_ratio"] > 0].sort_values("date").reset_index(drop=True)

    splits.to_csv(splits_file, index=False)
    print(f"Saved {len(splits)} split events to {splits_file.name}")

log_source("Stock split calendar", "https://finance.yahoo.com (via yfinance)", splits_file,
           "Used to distinguish genuine crashes from unadjusted splits.")

print(f"\nSplit events since {PRICE_START}: {len(splits)}")
print(f"Companies affected: {splits['ticker'].nunique()}\n")
print("Most recent:")
print(splits.tail(6).to_string(index=False))

Using cached split calendar (53 events)

Split events since 2023-01-01: 53
Companies affected: 47

Most recent:
      date ticker  split_ratio
2026-06-12   KLAC    10.000000
2026-06-24     DD     0.333333
2026-06-29    HON     0.953500
2026-07-01   SPGI     1.057000
2026-07-02   CRWD     4.000000
2026-08-11   MNST     2.000000


## 7. data and hypothesis relation

These are the characteristics the research question asks about.

| What we collect | usefulness |
|---|---|
| Total debt, debt-to-equity, cash | **H3** :do indebted companies suffer more when borrowing costs rise? |
| Free cash flow, operating cash flow | **H2** :are companies spending more than they earn more exposed? |
| Capital spending | **H2** : heavy spenders must keep raising money, which is now expensive |
| P/E, price-to-sales | **H1**: a high multiple means profits sit further in the future, where discounting bites hardest |
| Sector | **H4**: utilities and real estate are known rate proxies; banks may benefit |
| Market capitalisation, market beta | for Controling  we do not mistake "small" or "volatile" for "rate-sensitive" |
| Revenue growth, profit margin | for Controling  growth and quality shape valuation independently |
| Dividend yield | A high yield behaves like a bond, which should raise rate sensitivity |

**A caveat to state in the report.** These figures are a *snapshot taken today*, while rate
sensitivity is measured over 2023 onward. A company whose balance sheet changed sharply during
that window is imperfectly described by its current numbers. yfinance does not provide deep
history for these measures, so this is a genuine limitation of the design rather than an
oversight â€” it is disclosed rather than hidden.

In [21]:
# The Yahoo field name on the left, our clearer name on the right.
FUNDAMENTAL_FIELDS = {
    "marketCap": "market_cap",
    "trailingPE": "pe_ratio",
    "priceToSalesTrailing12Months": "price_to_sales",
    "priceToBook": "price_to_book",
    "debtToEquity": "debt_to_equity",
    "totalDebt": "total_debt",
    "totalCash": "total_cash",
    "freeCashflow": "free_cash_flow",
    "operatingCashflow": "operating_cash_flow",
    "totalRevenue": "revenue",
    "ebitda": "ebitda",
    "revenueGrowth": "revenue_growth",
    "profitMargins": "profit_margin",
    "returnOnEquity": "return_on_equity",
    "dividendYield": "dividend_yield",
    "beta": "market_beta",
    "sector": "sector",
    "industry": "industry",
}


def fetch_one_company(ticker):
    """Pull the fundamental fields for a single company.

    Yahoo does not carry every field for every company, so missing values are recorded
    as None rather than causing a failure. Any company that fails outright is recorded
    too as 500
    """
    record = {"ticker": ticker}
    try:
        info = yf.Ticker(ticker).info
        for yahoo_name, our_name in FUNDAMENTAL_FIELDS.items():
            record[our_name] = info.get(yahoo_name)
        record["download_ok"] = True
    except Exception as error:
        record["download_ok"] = False
        record["error"] = type(error).__name__
    return record


fundamentals_file = DATA_RAW / "company_fundamentals.csv"

if is_cached(fundamentals_file):
    fundamentals = pd.read_csv(fundamentals_file)
    print(f"Using cached fundamentals ({len(fundamentals)} companies)")
else:
    print(f"Downloading fundamentals for {len(tickers)} companies, one at a time.")

    records = []
    for position, ticker in enumerate(tickers, start=1):
        records.append(fetch_one_company(ticker))
        time.sleep(FUNDAMENTALS_PAUSE)   # stay comfortably inside Yahoo's rate limits

        if position % 50 == 0 or position == len(tickers):
            succeeded = sum(r["download_ok"] for r in records)
            print(f"   {position:3d}/{len(tickers)} companies â€” {succeeded} succeeded")

    fundamentals = pd.DataFrame(records)
    fundamentals.to_csv(fundamentals_file, index=False)
    print(f"\nSaved to {fundamentals_file.name}")

log_source("Company fundamentals", "https://finance.yahoo.com (via yfinance)",
           fundamentals_file, "Snapshot taken on the retrieval date; see limitation note.")

print(f"\nCompanies retrieved: {int(fundamentals['download_ok'].sum())} of {len(fundamentals)}")
fundamentals.head()

Using cached fundamentals (503 companies)

Companies retrieved: 503 of 503


,ticker,market_cap,pe_ratio,price_to_sales,price_to_book,debt_to_equity,total_debt,total_cash,free_cash_flow,operating_cash_flow,revenue,ebitda,revenue_growth,profit_margin,return_on_equity,dividend_yield,market_beta,sector,industry,download_ok
0,MMM,9.419670e+10,32.442272,3.740933,31.909502,437.937,1.316000e+10,5.303000e+09,6.350125e+09,4.899000e+09,2.518000e+10,6.488000e+09,0.025,0.11902,0.81892,1.71,1.080,Industrials,Conglomerates,True
1,AOS,8.449436e+09,17.317549,2.220672,4.587514,36.766,6.772000e+08,1.813000e+08,4.846125e+08,6.923000e+08,3.804900e+09,7.837000e+08,-0.007,0.13149,0.27133,2.32,1.157,Industrials,Specialty Industrial Machinery,True
2,ABT,1.925052e+11,36.003240,4.132342,3.766462,63.212,3.272000e+10,5.603000e+09,7.209125e+09,9.905000e+09,4.658500e+10,1.168100e+10,0.130,0.11645,0.10576,2.27,0.581,Healthcare,Medical Devices,True
3,ABBV,4.408251e+11,70.468925,6.846599,-74.266150,NaN,7.088500e+10,6.569000e+09,1.687025e+10,1.950700e+10,6.438600e+10,3.076300e+10,0.102,0.09800,NaN,2.77,0.280,Healthcare,Drug Manufacturers - General,True
4,ACN,1.082464e+11,14.128593,1.480787,3.393834,25.035,8.388770e+09,1.017157e+10,1.208938e+10,1.318209e+10,7.310059e+10,1.294377e+10,0.056,0.10656,0.24406,3.65,1.074,Technology,Information Technology Services,True


## 8. check missingness

Before moving on, we check how complete each field is. This matters for two reasons:
it tells us which hypotheses the data can actually support, and it is the raw material
for the missing-data audit in the next notebook.

A field missing for a handful of companies is fine. A field missing for a third of them
would quietly bias any result that uses it, so we would rather find out now.

In [22]:
measure_columns = [c for c in FUNDAMENTAL_FIELDS.values() if c not in ("sector", "industry")]

coverage = pd.DataFrame({
    "field": measure_columns,
    "present": [fundamentals[c].notna().sum() for c in measure_columns],
})
coverage["missing"] = len(fundamentals) - coverage["present"]
coverage["percent_present"] = (100 * coverage["present"] / len(fundamentals)).round(1)
coverage = coverage.sort_values("percent_present", ascending=False).reset_index(drop=True)

print(f"Field coverage across {len(fundamentals)} companies:\n")
print(coverage.to_string(index=False))

weak_fields = coverage[coverage["percent_present"] < 80]["field"].tolist()
if weak_fields:
    print(f"\nUnder 80% coverage, handle with care: {', '.join(weak_fields)}")
else:
    print("\nEvery field is present for at least 80% of companies.")

Field coverage across 503 companies:

              field  present  missing  percent_present
     revenue_growth      502        1             99.8
      profit_margin      502        1             99.8
            revenue      502        1             99.8
         total_cash      501        2             99.6
operating_cash_flow      501        2             99.6
         total_debt      501        2             99.6
      price_to_book      497        6             98.8
        market_beta      497        6             98.8
         market_cap      485       18             96.4
     price_to_sales      484       19             96.2
           pe_ratio      474       29             94.2
             ebitda      472       31             93.8
   return_on_equity      469       34             93.2
     free_cash_flow      469       34             93.2
     debt_to_equity      449       54             89.3
     dividend_yield      405       98             80.5

Every field is present for

## 9. source list for documentation

`SOURCES.txt` lists every URL and the date it was retrieved

In [23]:
sources_file = DATA_RAW / "SOURCES.txt"

lines = [
    "DATA SOURCES",
    "Project: Which company characteristics determine share price sensitivity to interest rates?",
    f"Collected on: {date.today().isoformat()}",
    f"Study window: {PRICE_START} to {PRICE_END}",
    "",
    "All sources are public and require no API key or account.",
    "=" * 78,
    "",
]

for entry in source_log:
    lines += [
        entry["dataset"],
        f"  URL           : {entry['url']}",
        f"  Saved as      : {entry['saved_as']}",
        f"  Retrieved on  : {entry['retrieved_on']}",
    ]
    if entry["note"]:
        lines.append(f"  Note          : {entry['note']}")
    lines.append("")

lines += [
    "=" * 78,
    "",
    "TO REPRODUCE",
    "  Run 01_data_collection.ipynb. Cached files in data/raw are reused; set",
    "  FORCE_REFRESH = True in the configuration cell to download everything again.",
    "",
    "REFERENCES",
    "  Fama, E. F., & French, K. R. (1993). Common risk factors in the returns on",
    "    stocks and bonds. Journal of Financial Economics, 33(1), 3-56.",
    "  Fama, E. F., & French, K. R. (2015). A five-factor asset pricing model.",
    "    Journal of Financial Economics, 116(1), 1-22.",
]

sources_file.write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote {sources_file.name} â€” {len(source_log)} sources documented\n")
print("\n".join(lines[:14]))

Wrote SOURCES.txt â€” 9 sources documented

DATA SOURCES
Project: Which company characteristics determine share price sensitivity to interest rates?
Collected on: 2026-08-16
Study window: 2023-01-01 to 2026-08-16

All sources are public and require no API key or account.

S&P 500 constituents
  URL           : https://en.wikipedia.org/wiki/List_of_S%26P_500_companies
  Saved as      : sp500_constituents.csv
  Retrieved on  : 2026-08-16
  Note          : Current index membership; survivorship bias noted in the report.



## 10. Summary

In [24]:
files = sorted(DATA_RAW.glob("*"))

print("Files in data/raw:\n")
for path in files:
    print(f"   {path.name:38s} {path.stat().st_size / 1024:8.1f} KB")

print(f"\n{'-' * 60}")
print("data we have")
print(f"{'-' * 60}")
print(f"Companies             : {len(constituents)} across {constituents['sector'].nunique()} sectors")
print(f"Daily price rows      : {len(prices):,}")
print(f"Price history         : {prices['date'].min().date()} to {prices['date'].max().date()}")
print(f"Rate observations     : {rates[RATE_FACTOR].notna().sum():,} days of {RATE_FACTOR}")
print(f"Factor rows           : {len(factors):,} days")
print(f"Fundamentals          : {int(fundamentals['download_ok'].sum())} companies")


Files in data/raw:

   SOURCES.txt                                 2.9 KB
   company_fundamentals.csv                  101.0 KB
   daily_prices.csv                        15154.1 KB
   fama_french_5factor_daily.csv             990.9 KB
   fred_DFII10.csv                            96.2 KB
   fred_DGS10.csv                            262.1 KB
   fred_DGS30.csv                            201.1 KB
   fred_T10YIE.csv                            95.3 KB
   sp500_constituents.csv                     29.8 KB
   stock_splits.csv                            1.1 KB

------------------------------------------------------------
data we have
------------------------------------------------------------
Companies             : 503 across 11 sectors
Daily price rows      : 452,228
Price history         : 2023-01-03 to 2026-08-14
Rate observations     : 5,908 days of DFII10
Factor rows           : 15,854 days
Fundamentals          : 503 companies
